# Evaluation Figures

This notebook reproduces **aggregate reporting figures** (coverage and performance summaries) from the CSV tables included in this repository.

**Privacy note:** This public repository does **not** include protected clinical data such as raw audio, participant-level labels, annotation exports, or participant-level model scores. As a result, some figures (e.g., ROC curves) cannot be regenerated exactly from public files alone.


## Configuration

Optionally, if you have a locally available (protected) matched participant-level table, you can point to it here to recompute the aggregate metrics.
If you do **not** have protected data, leave `PROTECTED_MATCHED_TABLE = None` and the notebook will use the aggregate metrics CSV shipped with this repository.


In [ ]:
from pathlib import Path

RUN_PROTECTED_LOCAL = True

# Point this to the folder that contains:
#   AUSC iPhone/
#   AUSC_Standardized/
#   Label/
PROTECTED_DATA_ROOT = None  # e.g., Path("/path/to/local/data_root")

METHODOLOGY_EXAMPLE_PARTICIPANT = "4023"
METHODOLOGY_TIME_WINDOW = (97.678, 8.0)  # (start_s, duration_s)


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def find_repo_root() -> Path:
    """Locate the repository root in a notebook-friendly way.

    In Jupyter, the kernel working directory can differ from the notebook file
    directory (e.g., if the server is started elsewhere). This helper tries a
    few common hints and then walks upward to find the project root.

    The repo root is identified by the presence of:
    - `src/evaluate_metrics.py`
    - `results/final_matched_metrics_table.csv`
    """

    def is_root(p: Path) -> bool:
        return (p / "src" / "evaluate_metrics.py").exists() and (p / "results" / "final_matched_metrics_table.csv").exists()

    candidates: list[Path] = []

    # 1) Current working directory
    candidates.append(Path.cwd().resolve())

    # 2) VS Code notebook path (if available)
    vsc = globals().get("__vsc_ipynb_file__")
    if vsc:
        try:
            candidates.append(Path(vsc).resolve().parent)
        except Exception:
            pass

    # 3) Jupyter session name (often relative to the server root)
    sess = os.environ.get("JPY_SESSION_NAME") or globals().get("__session__")
    if sess:
        try:
            candidates.append((Path.cwd() / Path(str(sess))).resolve().parent)
        except Exception:
            pass

    for start in candidates:
        p = start
        while True:
            if is_root(p):
                return p
            if p == p.parent:
                break
            p = p.parent

    raise RuntimeError(
        "Could not locate repository root.\n"
        "Tip: start Jupyter from the repository root, or set REPO_ROOT manually in this cell."
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from src.evaluate_metrics import (  # noqa: E402
    compute_method_metrics_table,
    confusion_matrix_from_metrics_row,
    validate_metric_table,
    compute_roc_curve,
)
# Example: Path('/path/to/protected/matched_table.csv')
PROTECTED_MATCHED_TABLE: Path | None = None

# Optional: PCG-only predictions export from the RR-feature pipeline (local-only).
PROTECTED_PCG_PREDICTIONS_CSV: Path | None = None


# Optional protected local inputs (NOT included in this repository).
PROTECTED_RAW_AUDIO_PATH: Path | None = None
PROTECTED_PREPROCESSED_AUDIO_PATH: Path | None = None
PROTECTED_ANNOTATION_CSV: Path | None = None
PROTECTED_ANNOTATION_KEY_CSV: Path | None = None

RUN_PROTECTED_LOCAL: bool = False

# Optional parameters for methodology examples (local-only).
# Example: METHODOLOGY_EXAMPLE_PARTICIPANT = "4023"
# Example: METHODOLOGY_TIME_WINDOW = (0.0, 8.0)  # start_s, duration_s
METHODOLOGY_EXAMPLE_PARTICIPANT: str | None = None
METHODOLOGY_TIME_WINDOW: tuple[float, float] | None = None

AGGREGATE_METRICS_CSV = REPO_ROOT / 'results' / 'final_matched_metrics_table.csv'
SAVE_REGENERATED_FIGURES: bool = False

OUT_DIR = REPO_ROOT / 'figures' / 'main' / 'regenerated'

if SAVE_REGENERATED_FIGURES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)


## Load metrics

- If a protected matched table is provided, recompute metrics using reusable functions from `src/evaluate_metrics.py`.
- Otherwise, load and validate the included aggregate metrics table.


In [ ]:
if PROTECTED_MATCHED_TABLE is not None and PROTECTED_MATCHED_TABLE.exists():
    matched = pd.read_csv(PROTECTED_MATCHED_TABLE)

    # These column names are expected in the protected matched table.
    # Update pred_col/prob_col to match your local export schema.
    methods = [
        {'method': 'PCG', 'pred_col': 'pcg_pred_4class', 'prob_col': 'pcg_paf'},
        {'method': 'FibriCheck iOS', 'pred_col': 'fibricheck_ios_4class'},
        {'method': 'FibriCheck Android', 'pred_col': 'fibricheck_android_4class'},
        {'method': 'Kardia', 'pred_col': 'kardia_4class'},
    ]

    df_metrics = compute_method_metrics_table(matched, methods=methods)
else:
    df_metrics = pd.read_csv(AGGREGATE_METRICS_CSV)

df_metrics = validate_metric_table(df_metrics)
df_metrics


## Figure 2: Coverage and AF/SR-classified performance

This figure is regenerated from the metrics table (protected-mode recomputation or public aggregate CSV).
The x-axis labels intentionally do not include sample-size text.


In [ ]:
# Ordering and display names for plotting
order = ['PCG', 'FibriCheck iOS', 'FibriCheck Android', 'Kardia']
display = {
    'PCG': 'PCG',
    'FibriCheck iOS': 'Fibri iOS',
    'FibriCheck Android': 'Fibri Android',
    'Kardia': 'Kardia',
}

df2 = df_metrics.set_index('method').loc[order].reset_index()
methods = [display[m] for m in df2['method'].tolist()]
n_ref = int(df2['n_ref'].iloc[0])

classified_n = df2['n_classified_AF_SR'].astype(int).to_numpy()
oa_n = df2['n_OA'].astype(int).to_numpy()
ui_n = df2['n_UI'].astype(int).to_numpy()
missing_n = df2['n_missing'].astype(int).to_numpy()

classified = classified_n / n_ref
oa = oa_n / n_ref
ui = ui_n / n_ref
missing = missing_n / n_ref

sens = df2['sensitivity'].astype(float).to_numpy()
spec = df2['specificity'].astype(float).to_numpy()
acc = df2['accuracy'].astype(float).to_numpy()

sns.set_theme(style='whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.8, 6.3))
fig.suptitle(
    'Coverage and AF/SR-classified performance vs ECG12 AF/SR reference',
    fontsize=20,
    fontweight='bold',
    y=0.98,
)

x = np.arange(len(methods))

# Panel A: coverage (stacked)
ax1.set_title('A. Coverage', fontsize=18, pad=16)
ax1.bar(x, classified, label='Classified AF/SR', color='#f6c667', edgecolor='white', linewidth=1.0)
ax1.bar(x, oa, bottom=classified, label='OA', color='#9ecae1', edgecolor='white', linewidth=1.0)
ax1.bar(x, ui, bottom=classified + oa, label='UI', color='#f28e8e', edgecolor='white', linewidth=1.0)
ax1.bar(x, missing, bottom=classified + oa + ui, label='Missing', color='#bdbdbd', edgecolor='white', linewidth=1.0)

ax1.set_ylim(0, 1.0)
ax1.set_xticks(x)
ax1.set_xticklabels(methods, fontsize=13)
ax1.tick_params(axis='x', pad=14)
ax1.set_ylabel('Proportion of cohort (ECG12 AF/SR with PCG available)', fontsize=15)

for i, n in enumerate(classified_n):
    pct = (n / n_ref) * 100
    y = max(0.06, classified[i] / 2)
    ax1.text(i, y, f'{n}/{n_ref}\nclassified\n({pct:.1f}%)', ha='center', va='center', fontsize=13, color='black')

ax1.legend(loc='upper right', frameon=True, fontsize=14)
fig.text(0.08, 0.06, f'Cohort: ECG12 AF/SR with PCG available (n={n_ref})', fontsize=14, color='#555555')

# Panel B: performance (grouped)
ax2.set_title('B. AF/SR-classified performance', fontsize=18, pad=16)
width = 0.24
ax2.bar(x - width, sens, width, label='Sensitivity', color='#4c78a8', edgecolor='white', linewidth=0.8)
ax2.bar(x, spec, width, label='Specificity', color='#59a14f', edgecolor='white', linewidth=0.8)
ax2.bar(x + width, acc, width, label='Accuracy', color='#f28e2b', edgecolor='white', linewidth=0.8)

ax2.set_ylim(0, 1.0)
ax2.set_xticks(x)
ax2.set_xticklabels(methods, fontsize=13)
ax2.tick_params(axis='x', pad=14)
ax2.set_ylabel('Metric value', fontsize=15)
ax2.legend(loc='lower right', frameon=True, fontsize=14)

plt.subplots_adjust(top=0.80, bottom=0.18, wspace=0.22)

plt.show()
if SAVE_REGENERATED_FIGURES:
    out_png = OUT_DIR / 'fig2_coverage_performance.png'
    fig.savefig(out_png, bbox_inches='tight', dpi=300)
    print('Wrote:', out_png.relative_to(REPO_ROOT))
plt.close(fig)


## Figure 3: PCG confusion matrix (aggregate)

The public repository includes enough aggregate counts (`tp/fp/tn/fn`) to reproduce the **PCG confusion matrix**.

**ROC note:** The ROC curve shown in the tracked Figure 3 was generated from participant-level probability scores (`pAF`), which are not included in this public repository. Therefore, this notebook does not attempt to reproduce the ROC curve from aggregate outputs.

If you provide a local protected matched table that contains participant-level PCG probability scores (e.g., `pcg_paf`), the notebook can additionally display the ROC curve and AUC in protected-local mode.


In [ ]:
# Confusion matrix is reproducible from aggregate counts.
pcg_row = df_metrics[df_metrics['method'] == 'PCG'].iloc[0]
cm = confusion_matrix_from_metrics_row(pcg_row)

sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(5.2, 4.6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, linewidths=0.0, ax=ax)
ax.set_xlabel('Predicted label')
ax.set_ylabel('ECG12 reference')
ax.set_xticklabels(['SR', 'AF'])
ax.set_yticklabels(['SR', 'AF'], rotation=0)
ax.set_title('PCG confusion matrix (AF/SR-classified subset)')

plt.show()
if SAVE_REGENERATED_FIGURES:
    out_png = OUT_DIR / 'fig3_pcg_confusion_matrix_regenerated.png'
    fig.savefig(out_png, bbox_inches='tight', dpi=300)
    print('Wrote:', out_png.relative_to(REPO_ROOT))
plt.close(fig)

# Optional: ROC curve requires participant-level probability scores.
# Optional: ROC curve requires participant-level probability scores.
# In public aggregate mode, participant-level scores are not included.

roc_source = None
roc_df = None

if PROTECTED_MATCHED_TABLE is not None and PROTECTED_MATCHED_TABLE.exists():
    roc_source = str(PROTECTED_MATCHED_TABLE)
    roc_df = pd.read_csv(PROTECTED_MATCHED_TABLE)
elif PROTECTED_PCG_PREDICTIONS_CSV is not None and PROTECTED_PCG_PREDICTIONS_CSV.exists():
    roc_source = str(PROTECTED_PCG_PREDICTIONS_CSV)
    roc_df = pd.read_csv(PROTECTED_PCG_PREDICTIONS_CSV)

if roc_df is not None:
    # Support a few common column schemas from local pipeline outputs.
    # Preferred (matched table): ecg12_4class, pcg_pred_4class, pcg_paf
    # RR pipeline exports: ecg12_4class, pcg_pred_oof / pcg_pred_full, pcg_paf_oof / pcg_paf_full
    if 'ecg12_4class' not in roc_df.columns:
        print('ROC skipped: missing ecg12_4class in', roc_source)
    else:
        if 'pcg_paf' in roc_df.columns:
            prob_col = 'pcg_paf'
        elif 'pcg_paf_oof' in roc_df.columns:
            prob_col = 'pcg_paf_oof'
        elif 'pcg_paf_full' in roc_df.columns:
            prob_col = 'pcg_paf_full'
        else:
            prob_col = None

        if 'pcg_pred_4class' in roc_df.columns:
            pred_col = 'pcg_pred_4class'
        elif 'pcg_pred_oof' in roc_df.columns:
            pred_col = 'pcg_pred_oof'
        elif 'pcg_pred_full' in roc_df.columns:
            pred_col = 'pcg_pred_full'
        else:
            pred_col = None

        if prob_col is None or pred_col is None:
            print('ROC skipped: missing required PCG columns in', roc_source)
            print('  need prob col in {pcg_paf, pcg_paf_oof, pcg_paf_full} and pred col in {pcg_pred_4class, pcg_pred_oof, pcg_pred_full}')
        else:
            sub = roc_df.copy()
            sub = sub[sub['ecg12_4class'].isin(['AF', 'SR'])]
            sub = sub[sub[pred_col].isin(['AF', 'SR'])]
            p = pd.to_numeric(sub[prob_col], errors='coerce')
            sub = sub[np.isfinite(p.to_numpy())].copy()

            if len(sub) >= 3 and len(set(sub['ecg12_4class'].tolist())) == 2:
                y_true = (sub['ecg12_4class'].astype(str).str.upper() == 'AF').astype(int).to_numpy()
                y_score = pd.to_numeric(sub[prob_col], errors='coerce').to_numpy(dtype=float)
                fpr, tpr, _thr, auc = compute_roc_curve(y_true, y_score)

                fig, ax = plt.subplots(figsize=(5.2, 4.6))
                ax.plot(fpr, tpr, lw=2.0, color='#4c78a8', label=f'AUC = {auc:.3f}')
                ax.plot([0, 1], [0, 1], ls='--', lw=1.2, color='#888888')
                ax.set_xlabel('False positive rate')
                ax.set_ylabel('True positive rate')
                ax.set_title(f'PCG ROC curve (n={len(sub)})')
                ax.legend(loc='lower right', frameon=False)
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1)
                plt.show()
                plt.close(fig)
            else:
                print('ROC skipped: not enough AF/SR samples after filtering in', roc_source)
else:
    print('No protected probability table configured; skipping ROC curve.')


## Methodology figures

To run it locally, set the `PROTECTED_*` paths and methodology parameters in the configuration cell above.

The plots below are **signal-derived** and should be displayed in the notebook only. Do not save or commit them.


In [ ]:
# Optional protected local inputs (NOT included in this repository).
#
# To use protected-local mode, set `RUN_PROTECTED_LOCAL = True` and fill in the
# paths below on your own machine. Do NOT commit those paths or any exported
# participant-level outputs to the public repository.

# Optional convenience: infer file paths from a single local data root.
# Set this to the folder that contains 'AUSC iPhone/', 'AUSC_Standardized/', and 'Label/'.
PROTECTED_DATA_ROOT = None  # e.g., Path('/path/to/local/data_root')
AUTO_INFER_METHOD_PATHS = True

PROTECTED_RAW_AUDIO_PATH = None  # e.g., Path('/path/to/raw.wav')
PROTECTED_PREPROCESSED_AUDIO_PATH = None  # e.g., Path('/path/to/preprocessed.wav')
PROTECTED_ANNOTATION_CSV = None  # e.g., Path('/path/to/Annotation.csv')
PROTECTED_ANNOTATION_KEY_CSV = None  # e.g., Path('/path/to/FileId_ParticipantId_Key.csv')

METHODOLOGY_EXAMPLE_PARTICIPANT = None  # e.g., '4023'
METHODOLOGY_TIME_WINDOW = None  # e.g., (start_s, duration_s)


In [ ]:
# If you set PROTECTED_DATA_ROOT, we can infer the expected files for the chosen participant.
# This avoids hard-coding absolute paths in the public notebook.
if RUN_PROTECTED_LOCAL and AUTO_INFER_METHOD_PATHS and PROTECTED_DATA_ROOT and METHODOLOGY_EXAMPLE_PARTICIPANT:
    from src.methodology_figures import infer_default_protected_paths
    paths = infer_default_protected_paths(data_root=PROTECTED_DATA_ROOT, participant_id=METHODOLOGY_EXAMPLE_PARTICIPANT)
    PROTECTED_RAW_AUDIO_PATH = paths['raw_wav']
    PROTECTED_PREPROCESSED_AUDIO_PATH = paths['preprocessed_wav']
    PROTECTED_ANNOTATION_CSV = paths['annotation_csv']
    PROTECTED_ANNOTATION_KEY_CSV = paths['annotation_key_csv']

if (RUN_PROTECTED_LOCAL and (
    PROTECTED_RAW_AUDIO_PATH
    and PROTECTED_PREPROCESSED_AUDIO_PATH
    and PROTECTED_ANNOTATION_CSV
    and PROTECTED_ANNOTATION_KEY_CSV
    and METHODOLOGY_EXAMPLE_PARTICIPANT
    and METHODOLOGY_TIME_WINDOW
)):
    from src.methodology_figures import (
        load_annotation_payload_for_participant,
        plot_preprocessing_example,
        plot_s1s2_annotation_example,
    )

    start_s, duration_s = METHODOLOGY_TIME_WINDOW

    fig = plot_preprocessing_example(
        raw_wav=PROTECTED_RAW_AUDIO_PATH,
        preprocessed_wav=PROTECTED_PREPROCESSED_AUDIO_PATH,
        start_s=float(start_s),
        duration_s=float(duration_s),
    )
    plt.show()

    payload = load_annotation_payload_for_participant(
        PROTECTED_ANNOTATION_CSV,
        PROTECTED_ANNOTATION_KEY_CSV,
        participant_id=METHODOLOGY_EXAMPLE_PARTICIPANT,
    )

    fig = plot_s1s2_annotation_example(
        preprocessed_wav=PROTECTED_PREPROCESSED_AUDIO_PATH,
        payload=payload,
        start_s=float(start_s),
        duration_s=float(duration_s),
    )
    plt.show()
else:
    print("Protected local paths not configured; skipping signal-derived methodology figures.")
